# AlgoTrade — BTC/USD Backtesting Lab

A lightweight backtesting notebook for crypto trading strategies, built on [`vectorbt`](https://github.com/polakowo/vectorbt).

**What's inside:**
- Data loading & preparation (1-minute BTC/USD OHLCV from Bitstamp)
- Technical indicators: SMA/EMA, RSI, MACD, Bollinger Bands, ATR
- Two backtested strategies: **Moving Average Crossover** and **RSI Mean Reversion**
- Performance metrics, equity curves, and a side-by-side strategy comparison

> Data source: `tutorial.csv` (or fetch fresh data with `data.ipynb`).

In [1]:
import numpy as np
import pandas as pd
import vectorbt as vbt
import datetime as dt

# Notebook-wide config
CSV_PATH = "tutorial.csv"
INIT_CASH = 10_000
FEES = 0.001       # 0.1% per trade, typical Bitstamp taker fee
FREQ = "1min"

vbt.settings.set_theme("dark")
vbt.settings["plotting"]["use_widgets"] = False  # plain static figures keep the notebook file small
vbt.settings["plotting"]["layout"]["width"] = 900
vbt.settings["plotting"]["layout"]["height"] = 450

## 1. Load & Prepare Data

Load 1-minute BTC/USD OHLCV data and index it by timestamp.

In [2]:
df = pd.read_csv(CSV_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.set_index("timestamp").sort_index()

open_, high, low, close, volume = df["open"], df["high"], df["low"], df["close"], df["volume"]

print(f"Rows: {len(df)}  |  Range: {df.index.min()} → {df.index.max()}")
df.head()

Rows: 1440  |  Range: 2026-08-15 00:00:00 → 2026-08-15 23:59:00


,open,high,low,close,volume
timestamp,,,,,
2026-08-15 00:00:00,62975.97,62975.97,62975.00,62975.00,0.010451
2026-08-15 00:01:00,62975.01,62975.97,62975.00,62975.97,0.041832
2026-08-15 00:02:00,62975.97,62991.91,62975.96,62991.91,0.455623
2026-08-15 00:03:00,62987.74,62987.74,62973.84,62973.84,0.010573
2026-08-15 00:04:00,62971.30,62971.30,62968.88,62968.88,0.020241


## 2. Technical Indicators

Compute the indicators the strategies below are built on:

- **SMA(10) / SMA(30)** — fast/slow trend-following moving averages
- **RSI(14)** — momentum oscillator for overbought/oversold conditions
- **MACD(12, 26, 9)** — trend + momentum
- **Bollinger Bands(20, 2σ)** — volatility bands
- **ATR(14)** — average true range, for volatility context

In [3]:
fast_ma = vbt.MA.run(close, window=10)
slow_ma = vbt.MA.run(close, window=30)
rsi = vbt.RSI.run(close, window=14)
macd = vbt.MACD.run(close, fast_window=12, slow_window=26, signal_window=9)
bb = vbt.BBANDS.run(close, window=20, alpha=2)
atr = vbt.ATR.run(high, low, close, window=14)

indicators = pd.DataFrame({
    "close": close,
    "sma_fast": fast_ma.ma,
    "sma_slow": slow_ma.ma,
    "rsi": rsi.rsi,
    "macd": macd.macd,
    "macd_signal": macd.signal,
    "bb_upper": bb.upper,
    "bb_lower": bb.lower,
    "atr": atr.atr,
})
indicators.tail()

,close,sma_fast,sma_slow,rsi,macd,macd_signal,bb_upper,bb_lower,atr
timestamp,,,,,,,,,
2026-08-15 23:55:00,63025.93,63033.119,63039.498333,0.175439,-3.975513,-5.014986,63042.571257,63027.832743,1.200206
2026-08-15 23:56:00,63025.93,63031.984,63038.675667,0.175593,-3.996026,-4.636161,63042.962041,63026.306959,1.040178
2026-08-15 23:57:00,63025.93,63030.849,63037.853000,0.175593,-4.015705,-4.361318,63043.110882,63025.022118,0.902821
2026-08-15 23:58:00,63025.93,63029.714,63037.030333,0.087873,-4.106538,-4.198269,63043.073090,63023.923910,0.783778
2026-08-15 23:59:00,63025.94,63028.580,63036.208000,0.175747,-4.196538,-4.138041,63042.877654,63022.984346,0.680608


In [4]:
fig = close.rename("Close").vbt.plot(trace_kwargs=dict(line=dict(color="lightskyblue")))
fast_ma.ma.rename("SMA 10").vbt.plot(trace_kwargs=dict(line=dict(color="orange", width=1)), fig=fig)
slow_ma.ma.rename("SMA 30").vbt.plot(trace_kwargs=dict(line=dict(color="magenta", width=1)), fig=fig)
bb.upper.rename("BB Upper").vbt.plot(trace_kwargs=dict(line=dict(color="gray", width=1, dash="dot")), fig=fig)
bb.lower.rename("BB Lower").vbt.plot(trace_kwargs=dict(line=dict(color="gray", width=1, dash="dot")), fig=fig)
fig.update_layout(title="BTC/USD — Price with SMA & Bollinger Bands")
fig.show()

## 3. Strategy 1 — Moving Average Crossover

**Rule:** go long when SMA(10) crosses above SMA(30) (bullish momentum); exit when it crosses back below.

A classic trend-following strategy — profits in sustained trends, whipsaws in choppy/sideways markets.

In [5]:
ma_entries = fast_ma.ma_crossed_above(slow_ma)
ma_exits = fast_ma.ma_crossed_below(slow_ma)

pf_ma = vbt.Portfolio.from_signals(
    close, ma_entries, ma_exits,
    init_cash=INIT_CASH, fees=FEES, freq=FREQ,
)

pf_ma.stats()

Start                               2026-08-15 00:00:00
End                                 2026-08-15 23:59:00
Period                                  1 days 00:00:00
Start Value                                     10000.0
End Value                                   9641.810195
Total Return [%]                              -3.581898
Benchmark Return [%]                           0.080889
Max Gross Exposure [%]                            100.0
Total Fees Paid                              392.788323
Max Drawdown [%]                               3.581898
Max Drawdown Duration                   0 days 22:44:00
Total Trades                                         20
Total Closed Trades                                  20
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                        0.0
Best Trade [%]                                -0.088251
Worst Trade [%]                               -0

In [6]:
pf_ma.plot(title="MA Crossover — Portfolio Value, Trades & Drawdown").show()

## 4. Strategy 2 — RSI Mean Reversion

**Rule:** go long when RSI(14) crosses above 30 (exiting oversold); exit when RSI crosses above 70 (overbought).

A contrarian strategy — profits in range-bound/choppy markets, underperforms in strong trends.

In [7]:
RSI_LOWER, RSI_UPPER = 30, 70

rsi_entries = rsi.rsi_crossed_above(RSI_LOWER)
rsi_exits = rsi.rsi_crossed_above(RSI_UPPER)

pf_rsi = vbt.Portfolio.from_signals(
    close, rsi_entries, rsi_exits,
    init_cash=INIT_CASH, fees=FEES, freq=FREQ,
)

pf_rsi.stats()

Start                               2026-08-15 00:00:00
End                                 2026-08-15 23:59:00
Period                                  1 days 00:00:00
Start Value                                     10000.0
End Value                                   9601.439933
Total Return [%]                              -3.985601
Benchmark Return [%]                           0.080889
Max Gross Exposure [%]                            100.0
Total Fees Paid                              382.298797
Max Drawdown [%]                               3.985616
Max Drawdown Duration                   0 days 23:43:00
Total Trades                                         20
Total Closed Trades                                  19
Total Open Trades                                     1
Open Trade PnL                               -15.946691
Win Rate [%]                                        0.0
Best Trade [%]                                 -0.17481
Worst Trade [%]                               -0

In [8]:
pf_rsi.plot(title="RSI Mean Reversion — Portfolio Value, Trades & Drawdown").show()

## 5. Strategy Comparison

Compare both strategies against each other and a buy & hold benchmark.

In [9]:
pf_hold = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, freq=FREQ)

metrics = ["total_return", "sharpe_ratio", "max_dd", "win_rate", "total_trades"]

comparison = pd.DataFrame({
    "Buy & Hold": pf_hold.stats(metrics=metrics),
    "MA Crossover": pf_ma.stats(metrics=metrics),
    "RSI Mean Reversion": pf_rsi.stats(metrics=metrics),
}).T

comparison

,Total Return [%],Sharpe Ratio,Max Drawdown [%],Win Rate [%],Total Trades
Buy & Hold,0.080889,4.289101,0.393697,NaN,1.0
MA Crossover,-3.581898,-102.899273,3.581898,0.0,20.0
RSI Mean Reversion,-3.985601,-122.340308,3.985616,0.0,20.0


In [10]:
comparison["Total Return [%]"].vbt.barplot(
    trace_kwargs=dict(marker_color=["#888888", "#f4a261", "#2a9d8f"]),
).update_layout(title="Total Return by Strategy", yaxis_title="Return [%]").show()

## 6. Takeaways & Next Steps

- Both strategies are tested here on a single day of 1-minute data — too short to draw real conclusions. Pull a longer history with `data.ipynb` (increase `limit`/`step`, or paginate) before trusting the numbers.
- Trading fees materially eat into short-timeframe strategies with many trades — compare `Total Fees Paid` against `Total Return`.
- Ideas to extend this notebook:
  - Parameter sweeps (e.g. grid search MA windows or RSI thresholds with `vbt.Portfolio.from_signals` on broadcast arrays)
  - Add stop-loss / take-profit (`sl_stop`, `tp_stop` in `from_signals`)
  - Walk-forward / train-test split to check for overfitting
  - Combine signals (e.g. only take MA crossover trades when RSI confirms momentum)